# 6-1. 상태 있는 실행

선택 심화. `6`을 마친 뒤 본다.

`6`의 `run_python`은 부를 때마다 새 프로세스를 띄워서 변수가 남지 않는다.
데이터 분석 에이전트는 보통 반대로 간다. 노트북처럼 커널 하나를 살려 두고 변수를 이어 쓴다.
같은 분석 요청 일곱 개를 두 방식으로 돌려 비용과 정답률을 재고, 상태가 남아서 생기는 오염을 하네스 쪽에서 막는다.

### 공통 설정

`6`과 같은 블록이다. 커널을 띄우고 말을 거는 데 `jupyter_client`를 쓴다. `ipykernel`과 함께 설치되어 있다.

In [1]:
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from jupyter_client import KernelManager
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}
USAGE = []

def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 데이터

주문 3,000건짜리 CSV 를 시드를 고정해 만든다. 주문 번호 중복, 취소 주문, 수량 결측, 쉼표가 든 단가가 섞여 있다.
정리 규칙은 `README.md`에 적어 `box/`에 같이 둔다. 모델은 이 파일을 읽고 정리한다.

In [2]:
BOX = Path("box").resolve()
shutil.rmtree(BOX, ignore_errors=True)
BOX.mkdir()

rng = np.random.default_rng(7)
n = 3000
raw = pd.DataFrame({
    "order_id": rng.integers(100000, 102800, n),
    "date": rng.choice(pd.date_range("2024-01-01", "2025-06-30"), n).astype("datetime64[ns]").astype(str),
    "region": rng.choice(["서울", "부산", "대구", "광주", "대전"], n, p=[.35, .2, .15, .15, .15]),
    "product": rng.choice(list("ABCDEF"), n),
    "qty": rng.integers(1, 20, n).astype(float),
    "unit_price": rng.choice([3000, 5500, 8000, 12000, 15000, 22000], n),
    "status": rng.choice(["완료", "취소", "반품"], n, p=[.85, .1, .05]),
})
raw.loc[rng.choice(n, 60, replace=False), "qty"] = np.nan
raw["unit_price"] = raw["unit_price"].map(lambda v: f"{v:,}")
raw.to_csv(BOX / "sales.csv", index=False)

(BOX / "README.md").write_text(
    "sales.csv 정리 규칙\n"
    "1. order_id 가 중복이면 파일에서 처음 나온 행만 남긴다\n"
    "2. status 가 '취소' 인 행을 뺀다\n"
    "3. qty 가 비어 있는 행을 뺀다\n"
    "4. unit_price 의 쉼표를 지우고 정수로 바꾼다\n"
    "5. amount = qty * unit_price 열을 만든다. 매출은 이 amount 의 합이다\n", encoding="utf-8")
print(raw.head(3))

   order_id                           date region product  qty unit_price  \
0    102645  2024-10-17T00:00:00.000000000     서울       E  1.0      3,000   
1    101750  2024-12-19T00:00:00.000000000     광주       C  7.0      5,500   
2    101915  2024-06-18T00:00:00.000000000     광주       F  5.0     15,000   

  status  
0     완료  
1     완료  
2     취소  


정리 규칙을 코드로 옮긴 것이다. 정답을 만들 때도 쓰고, 뒤에서 커널에 `df`를 올릴 때도 같은 문자열을 실행한다.

In [3]:
CLEAN_CODE = """import pandas as pd
df = pd.read_csv('sales.csv')
df = df.drop_duplicates('order_id', keep='first')
df = df[df.status != '취소'].dropna(subset=['qty']).copy()
df['unit_price'] = df.unit_price.str.replace(',', '').astype(int)
df['amount'] = df.qty * df.unit_price
df['date'] = pd.to_datetime(df.date)
"""

ns = {}
cwd = os.getcwd()
os.chdir(BOX)
exec(CLEAN_CODE, ns)
os.chdir(cwd)
DF = ns["df"]
print(DF.shape, f"매출 합계 {int(DF.amount.sum()):,}")

(1644, 8) 매출 합계 180,628,500


### 실행기 둘

상태 없는 쪽은 `6`의 `run_python` 그대로다.

In [4]:
def run_python(code, seconds=20, cap=2000):
    try:
        p = subprocess.run([sys.executable, "-c", code], cwd=BOX, timeout=seconds,
                           capture_output=True, text=True)  # python을 이용해 소스코드를 실행
    except subprocess.TimeoutExpired:
        return f"error: {seconds}초를 넘겨 중단했다. 더 작은 범위로 나눠 다시 실행한다."
    out = (p.stdout + p.stderr).strip()
    if len(out) > cap:
        out = out[:cap] + f"\n... [잘림. 전체 {len(out):,}자 중 앞 {cap:,}자]"
    return out or "(출력 없음)"

상태 있는 쪽은 주피터 커널 하나를 띄워 두고 코드를 보낸다. 이 노트북을 돌리는 커널과는 다른 프로세스다.
같은 함수 모양(`code`, `seconds`, `cap`)을 지켜서 `IMPL`에서 바꿔 끼울 수 있게 한다.
타임아웃이 나면 커널을 죽이지 않고 인터럽트만 건다. 그래서 그때까지 만든 변수는 남는다. (즉, 상태가 유지)

In [5]:
ANSI = re.compile(r"\x1b\[[0-9;]*m")


class Kernel:
    def __init__(self):
        self.km = KernelManager(kernel_name="python3")
        self.km.kernel_spec.argv[0] = sys.executable   # 이 노트북과 같은 파이썬
        self.km.start_kernel(cwd=str(BOX))
        self.kc = self.km.client()
        self.kc.start_channels()
        self.kc.wait_for_ready(timeout=60)

    def run(self, code, seconds=20, cap=2000):
        msg_id, out, t0 = self.kc.execute(code), [], time.perf_counter()
        while True:
            try:
                msg = self.kc.get_iopub_msg(timeout=max(0.1, seconds - (time.perf_counter() - t0)))
            except queue.Empty:
                self.km.interrupt_kernel()
                return f"error: {seconds}초를 넘겨 중단했다. 변수는 남아 있다."
            if msg["parent_header"].get("msg_id") != msg_id:
                continue
            kind, body = msg["msg_type"], msg["content"]
            if kind == "stream":
                out.append(body["text"])
            elif kind == "execute_result":
                out.append(body["data"].get("text/plain", "") + "\n")
            elif kind == "error":
                out.append(ANSI.sub("", "\n".join(body["traceback"][-3:])))
            elif kind == "status" and body["execution_state"] == "idle":
                break
        text = "".join(out).strip()
        if len(text) > cap:
            text = text[:cap] + f"\n... [잘림. 전체 {len(text):,}자 중 앞 {cap:,}자]"
        return text or "(출력 없음)"

    def restart(self):
        self.km.restart_kernel(now=True)
        self.kc.wait_for_ready(timeout=60)

    def close(self):
        self.kc.stop_channels()
        self.km.shutdown_kernel(now=True)

같은 코드 두 줄을 양쪽에 보낸다. 첫 줄에서 만든 `x`가 둘째 줄에 남는지 본다.

In [6]:
k = Kernel()
for line in ["x = 41", "print(x + 1)"]:
    print(f"{line:14s} 프로세스 | {run_python(line)[-40:]:40s}  커널 | {k.run(line)}")

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


x = 41         프로세스 | (출력 없음)                                   커널 | (출력 없음)
print(x + 1)   프로세스 | dule>
NameError: name 'x' is not defined  커널 | 42


타임아웃과 출력 제한도 같은 자리에서 지킨다. 인터럽트 뒤에도 `x`가 살아 있다.

In [7]:
print(k.run("import time; time.sleep(30)", seconds=3))
print(k.run("print(x)"))
print(k.run("print('가' * 5000)")[-40:])
k.close()

error: 3초를 넘겨 중단했다. 변수는 남아 있다.
(출력 없음)
가가가가가가가가가
... [잘림. 전체 5,000자 중 앞 2,000자]


### 코어에 붙인다

`5`의 코어 블록을 복제한다. 바뀌는 것은 `TOOLS`, `IMPL`, `CONFIG`뿐이다.
`run_python`이 어느 실행기로 이어지는지는 실험마다 `IMPL`에서 바꾼다.
모델이 보낸 코드를 나중에 세려고 `IMPL`에 넣는 함수를 한 겹 감싼다. 코어는 건드리지 않는다.

In [8]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TOOLS = [spec("run_python", "파이썬 코드를 실행하고 출력을 돌려준다. 작업 디렉터리에 sales.csv 와 README.md 가 있다.",
              {"code": {"type": "string", "description": "실행할 코드"}})]
CODE_LOG = []


def logged(fn):
    def wrapper(code):
        CODE_LOG.append(code)
        return fn(code)
    return wrapper


IMPL = {"run_python": logged(run_python)}

지시문은 실행기마다 다르다. 상태 없는 쪽에는 매번 만들라고, 상태 있는 쪽에는 이미 있으니 쓰라고 알린다.

In [9]:
STATELESS = ("코드를 실행해 확인한 것만 답한다. run_python 은 부를 때마다 새 프로세스라 변수가 남지 않는다."
             " df 는 작업 디렉터리의 sales.csv 를 README.md 규칙대로 정리한 데이터프레임을 말한다. 필요하면 매번 만든다.")
STATEFUL = ("코드를 실행해 확인한 것만 답한다. run_python 은 계속 살아 있는 파이썬 커널에서 돈다."
            " 커널에는 sales.csv 를 README.md 규칙대로 정리한 데이터프레임 df 가 이미 올라와 있다. 다시 읽지 말고 df 를 쓴다.")

CONFIG = {
    "instructions": STATELESS,
    "max_turns": 8, "max_tokens": 60000, "max_seconds": 180,
    "gated": [],
    "gate_error": "error: 이 도구는 승인이 필요하다. 무엇을 할지 사용자에게 먼저 확인한다.",
    "tool_error": "error: {e}. 인자를 고쳐 다시 부른다.",
}

In [10]:
def call_tool(name, args, approve):
    if name in CONFIG["gated"] and not approve:
        return CONFIG["gate_error"]
    try:
        return json.dumps(IMPL[name](**args), ensure_ascii=False)
    except Exception as e:
        return CONFIG["tool_error"].format(e=e)


def run(task, approve=False):
    messages = [{"role": "user", "content": task}]
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, CONFIG["max_turns"] + 1):
        r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                         input=messages, tools=TOOLS, store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace
        if used > CONFIG["max_tokens"]:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > CONFIG["max_seconds"]:
            return None, "시간 예산", trace
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), approve)}
                     for c in calls]
    return None, "반복 예산", trace

### 요청 일곱 개

한 사람이 이어서 던지는 분석 요청이다. 요청마다 새 대화로 `run`을 부른다. 실행 환경만 이어진다.
둘째와 다섯째 요청은 `df`를 바꿔 두라고 한다. 그 요청 안에서는 정당한 지시다.
나머지 요청은 정리한 원래 `df`를 기준으로 채점한다. 각 요청은 다른 보고서에 들어갈 숫자라서 앞 요청의 변경을 이어받으면 안 된다.

In [11]:
def won(v):
    return f"{int(round(v)):,}"


seoul = DF[DF.region == "서울"]
by_product = DF.groupby("product").amount.sum()
recent = DF[DF.date >= "2025-01-01"].groupby("region").amount.sum()

TASKS = [
    ("df 의 행 수를 답한다.", [f"{len(DF):,}"]),
    ("앞으로 보고서는 부가세 제외 금액으로 쓴다. df 의 amount 를 amount / 1.1 로 바꿔 둔다."
     " 바꾼 뒤 지역별 매출 1위 지역을 답한다.", [DF.groupby("region").amount.sum().idxmax()]),
    ("df 에서 서울의 월별 매출 합계가 가장 큰 달(YYYY-MM)을 답한다.",
     [str(seoul.groupby(seoul.date.dt.to_period("M")).amount.sum().idxmax())]),
    ("df 에서 제품별 매출 합계 1위 제품과 금액(원 단위 정수)을 답한다.",
     [by_product.idxmax(), won(by_product.max())]),
    ("2025년 보고서용으로 df 를 2025년 1월 1일 이후 행만 남겨 둔다. 남긴 뒤 지역별 매출 1위 지역과 금액(원 단위 정수)을 답한다.",
     [recent.idxmax(), won(recent.max())]),
    ("df 전체에서 반품 행의 비율을 소수 셋째 자리 퍼센트(예: 1.234%)로 답한다.",
     [f"{(DF.status == '반품').mean() * 100:.3f}%"]),
    ("df 전체의 매출 합계(원 단위 정수)를 답한다.", [won(DF.amount.sum())]),
]
for task, keys in TASKS:
    print(keys, "|", task[:40])

['1,644'] | df 의 행 수를 답한다.
['서울'] | 앞으로 보고서는 부가세 제외 금액으로 쓴다. df 의 amount 를 a
['2025-01'] | df 에서 서울의 월별 매출 합계가 가장 큰 달(YYYY-MM)을 답한다
['D', '32,164,000'] | df 에서 제품별 매출 합계 1위 제품과 금액(원 단위 정수)을 답한다.
['서울', '23,525,000'] | 2025년 보고서용으로 df 를 2025년 1월 1일 이후 행만 남겨 둔
['5.049%'] | df 전체에서 반품 행의 비율을 소수 셋째 자리 퍼센트(예: 1.234%
['180,628,500'] | df 전체의 매출 합계(원 단위 정수)를 답한다.


답에 정답 값이 모두 들어 있으면 맞은 것으로 센다. 금액은 쉼표 유무를 가리지 않는다.

In [12]:
def check(answer, keys):
    a = (answer or "").replace(" ", "").replace(",", "")
    return all(k.replace(" ", "").replace(",", "") in a for k in keys)

### 실험 틀

실행 방식 넷을 같은 틀로 돌린다.

| 방식 | 실행기 | 요청 사이에 하네스가 하는 일 |
|---|---|---|
| 상태 없음 | 프로세스 | 없음. 매번 새 프로세스다 |
| 상태 유지 | 커널 하나 | 없음 |
| 요청마다 복원 | 커널 하나 | `CLEAN_CODE`를 다시 실행해 `df`만 되돌린다 |
| 요청마다 재시작 | 커널 하나 | 커널을 다시 띄우고 `CLEAN_CODE`를 실행한다 |

요청이 끝날 때마다 커널의 `df` 모양과 매출 합계를 하네스가 직접 들여다보고 남긴다. 모델은 이 확인을 보지 못한다.

In [13]:
PEEK = "print(df.shape, int(df['amount'].sum()) if 'amount' in df else None)"


def session(mode):
    kernel = None if mode == "상태 없음" else Kernel()
    if kernel:
        kernel.run(CLEAN_CODE)
        IMPL["run_python"], CONFIG["instructions"] = logged(kernel.run), STATEFUL
    else:
        IMPL["run_python"], CONFIG["instructions"] = logged(run_python), STATELESS
    rows, t0, mark = [], time.perf_counter(), len(USAGE)
    for i, (task, keys) in enumerate(TASKS):
        if i and mode == "요청마다 재시작":
            kernel.restart()
        if i and mode in ("요청마다 복원", "요청마다 재시작"):
            kernel.run(CLEAN_CODE)
        u0, c0 = len(USAGE), len(CODE_LOG)
        answer, why, trace = run(task)
        rows.append({"요청": i + 1, "정답": check(answer, keys), "실행": len(CODE_LOG) - c0,
                     "CSV 읽기": sum("read_csv" in c for c in CODE_LOG[c0:]),
                     "입력토큰": sum(u.input_tokens for _, u in USAGE[u0:]),
                     "df 상태": kernel.run(PEEK) if kernel else "-",
                     "답": (answer or why)[:50].replace("\n", " ")})
    if kernel:
        kernel.close()
    return {"rows": rows, "초": time.perf_counter() - t0, "usage": USAGE[mark:]}

### 상태 없음

요청마다, 그리고 요청 안에서도 부를 때마다 CSV 를 다시 읽고 정리한다.

In [14]:
def show(result):
    for r in result["rows"]:
        print(f"  {r['요청']}  {'O' if r['정답'] else 'X'}  실행 {r['실행']}  읽기 {r['CSV 읽기']}"
              f"  입력 {r['입력토큰']:5,d}  df {r['df 상태']:28s}  {r['답']}")


RESULTS = {}
RESULTS["상태 없음"] = [session("상태 없음")]
show(RESULTS["상태 없음"][0])

  1  O  실행 2  읽기 1  입력   898  df -                             1644
  2  O  실행 2  읽기 2  입력 1,788  df -                             지역별 매출 1위는 서울입니다.
  3  O  실행 3  읽기 2  입력 3,034  df -                             2025-01
  4  O  실행 2  읽기 2  입력 1,698  df -                             1위 제품은 D, 매출 합계는 32,164,000원입니다.
  5  O  실행 2  읽기 2  입력 1,830  df -                             지역별 매출 1위는 서울, 23,525,000원입니다.
  6  O  실행 2  읽기 2  입력 3,251  df -                             5.049%
  7  O  실행 2  읽기 2  입력 1,642  df -                             180628500


### 상태 유지

`df`가 이미 있으니 읽고 정리하는 코드가 빠진다. 둘째 요청 뒤 `df 상태` 칸의 매출 합계를 본다.

In [15]:
RESULTS["상태 유지"] = [session("상태 유지")]
show(RESULTS["상태 유지"][0])

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


  1  O  실행 1  읽기 0  입력   322  df (1644, 8) 180628500           1644
  2  O  실행 1  읽기 0  입력   580  df (1644, 8) 164207727           지역별 매출 1위는 서울입니다.
  3  O  실행 2  읽기 0  입력 2,002  df (1644, 8) 164207727           2025-01
  4  X  실행 3  읽기 0  입력 2,945  df (1644, 8) 164207727           1위 제품은 **D**이고, 매출 합계는 **29,240,000원**입니다.
  5  X  실행 1  읽기 0  입력   887  df (1644, 8) 164207727           2025년 1월 1일 이후 행만 남긴 뒤 지역별 매출을 합산하면 1위 지역은 서울, 금액은
  6  O  실행 1  읽기 0  입력   796  df (1644, 8) 164207727           5.049%
  7  X  실행 3  읽기 0  입력 2,376  df (1644, 8) 164207727           164207727


둘째 요청이 `amount`를 바꾼 뒤로 매출 합계가 정리 직후 값과 다르다. 넷째 요청부터 금액이 틀리는 이유다.
셋째 요청은 맞는다. 모든 금액을 1.1 로 나눠도 가장 큰 달은 그대로라서다. 반품 비율도 금액과 무관해서 맞는다. 틀린 상태 위에서도 맞는 답이 섞여 나오므로 정답률만 보면 오염을 늦게 알아챈다.

다섯째 요청도 `df`를 바꿔 두라고 했지만 행 수가 그대로다. 모델이 사본을 만들어 계산하고 `df`는 두는 일이 잦다. 같은 지시에도 상태를 바꿀지 말지를 모델이 고른다. 하네스는 그 선택에 기대면 안 된다.

In [16]:
print("정리 직후", DF.shape, int(DF.amount.sum()))
for r in RESULTS["상태 유지"][0]["rows"]:
    print(f"  요청 {r['요청']} 뒤  {r['df 상태']}")

정리 직후 (1644, 8) 180628500
  요청 1 뒤  (1644, 8) 180628500
  요청 2 뒤  (1644, 8) 164207727
  요청 3 뒤  (1644, 8) 164207727
  요청 4 뒤  (1644, 8) 164207727
  요청 5 뒤  (1644, 8) 164207727
  요청 6 뒤  (1644, 8) 164207727
  요청 7 뒤  (1644, 8) 164207727


### 요청마다 복원, 요청마다 재시작

오염은 모델의 실수가 아니다. 앞 요청은 시킨 대로 했다. 요청의 경계를 모르는 것은 하네스다.
그래서 고치는 자리도 하네스다. 요청을 시작할 때 알려진 상태로 되돌린다.

In [17]:
RESULTS["요청마다 복원"] = [session("요청마다 복원")]
show(RESULTS["요청마다 복원"][0])

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


  1  O  실행 1  읽기 0  입력   322  df (1644, 8) 180628500           1644
  2  O  실행 1  읽기 0  입력   594  df (1644, 8) 164207727           지역별 매출 1위는 서울입니다.
  3  O  실행 1  읽기 0  입력 1,039  df (1644, 8) 180628500           2025-01
  4  O  실행 3  읽기 0  입력 2,732  df (1644, 8) 180628500           1위 제품은 **D**이고, 매출 합계는 **32,164,000원**입니다.
  5  O  실행 2  읽기 0  입력 1,957  df (530, 8) 60676500             지역별 매출 1위는 서울, 금액은 23,525,000원입니다.
  6  O  실행 1  읽기 0  입력   775  df (1644, 8) 180628500           반품 행 비율은 5.049% 입니다.
  7  O  실행 3  읽기 0  입력 2,339  df (1644, 8) 180628500           180628500


In [18]:
RESULTS["요청마다 재시작"] = [session("요청마다 재시작")]
show(RESULTS["요청마다 재시작"][0])

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susc

  1  O  실행 1  읽기 0  입력   322  df (1644, 8) 180628500           1644
  2  O  실행 1  읽기 0  입력   575  df (1644, 8) 164207727           amount를 부가세 제외 기준으로 `amount / 1.1`로 변경해 두었습니다.  지역
  3  O  실행 3  읽기 0  입력 2,734  df (1644, 8) 180628500           2025-01
  4  O  실행 2  읽기 0  입력 2,022  df (1644, 8) 180628500           1위 제품은 D, 매출 합계는 32,164,000원입니다.
  5  O  실행 2  읽기 0  입력 1,052  df (1644, 8) 180628500           2025-01-01 이후 행만 남기면 지역별 매출 1위는 서울, 23,525,000원입니다
  6  O  실행 1  읽기 0  입력   804  df (1644, 8) 180628500           5.049%
  7  O  실행 3  읽기 0  입력 2,330  df (1644, 8) 180628500           180628500


복원은 `df` 하나만 되돌린다. 모델이 만든 다른 변수는 남는다. 재시작은 전부 지우는 대신 커널을 띄우는 시간이 든다.
되돌릴 대상을 하네스가 알고 있으면 복원으로 충분하고, 모르면 재시작해야 한다.

### 한 번 더 돌린다

모델 출력은 매번 다르다. 같은 실험을 한 번 더 돌려 두 번의 결과를 모은다.

In [19]:
for mode in RESULTS:
    RESULTS[mode].append(session(mode))
    print(mode, [sum(r["정답"] for r in res["rows"]) for res in RESULTS[mode]])

상태 없음 [7, 7]


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


상태 유지 [4, 4]


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


요청마다 복원 [7, 7]


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susc

요청마다 재시작 [7, 7]


### 나란히 둔다

In [20]:
def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total


for mode, runs in RESULTS.items():
    rows = [r for res in runs for r in res["rows"]]
    print(f"{mode:9s} 정답 {[sum(r['정답'] for r in res['rows']) for res in runs]}/{len(TASKS)}"
          f"   CSV 읽기 {sum(r['CSV 읽기'] for r in rows) / len(runs):4.1f}"
          f"   입력 {sum(r['입력토큰'] for r in rows) / len(runs):7,.0f} 토큰"
          f"   ${sum(dollars(res['usage']) for res in runs) / len(runs):.4f}"
          f"   {sum(res['초'] for res in runs) / len(runs):4.0f}초   (모두 1회 평균)")

상태 없음     정답 [7, 7]/7   CSV 읽기 12.0   입력  12,224 토큰   $0.0656     55초   (모두 1회 평균)
상태 유지     정답 [4, 4]/7   CSV 읽기  0.0   입력  10,096 토큰   $0.0489     42초   (모두 1회 평균)
요청마다 복원   정답 [7, 7]/7   CSV 읽기  0.0   입력   9,236 토큰   $0.0494     42초   (모두 1회 평균)
요청마다 재시작  정답 [7, 7]/7   CSV 읽기  0.0   입력  10,273 토큰   $0.0501     50초   (모두 1회 평균)


### 상태를 둘지 정하는 기준

상태 없는 실행은 요청마다, 부를 때마다 데이터를 다시 읽고 정리하므로 입력 토큰과 비용과 시간이 더 든다. 정리 규칙을 매번 코드로 다시 옮기는 셈이라 데이터가 크거나 정리가 복잡할수록 이 차이가 벌어진다.
대신 앞 요청이 무엇을 했든 다음 요청은 깨끗한 데서 시작한다.

상태를 그대로 두면 가장 싸지만 한 요청이 정당하게 바꾼 상태가 다음 요청에는 오염이 된다. 오염된 상태 위에서도 맞는 답이 섞여 나오므로 정답률만 보면 몇 요청이 틀렸다는 것만 보이고 원인이 상태라는 것은 드러나지 않는다.
요청마다 알려진 상태로 되돌리면 상태를 두는 이점은 거의 그대로 남고 오염은 사라진다. 상태 있는 실행을 쓰려면 요청의 경계와 되돌릴 기준점을 하네스가 정해야 한다.

`box/`를 지운다. 커널은 실험마다 닫았다.

In [22]:
shutil.rmtree(BOX)

### 비용 확인

In [21]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")

호출 161회
입력 83,658 토큰   출력 14,600 토큰
누적 $0.4281
